## 1. Imports & Load Data

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [19]:
df = pd.read_csv('dataset/diabetic_data.csv')
y_target = 'readmitted'
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 2. Handle Missing / Invalid Values

In [20]:
feature_string = df.select_dtypes(include="object").columns.to_list()
df[feature_string] = df[feature_string].replace("?", np.nan)

In [21]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 374017
Duplicates: 0
Missing Feature:
['race', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult']


In [22]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
['race', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult']


## 3. Handling Duplicated

In [23]:
df = df.drop_duplicates()

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [24]:
feature_numerik = df.select_dtypes(include=np.number).columns

Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 45097
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

### 5.1 Drop Feature

In [25]:
df = df.drop(columns=[
    'encounter_id', 'patient_nbr', 'race', 'weight', 'admission_source_id',
    'age', 'number_diagnoses', 'payer_code', 'admission_type_id',
    'discharge_disposition_id', 'medical_specialty'],errors='ignore')
df.head(2)

,gender,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
1,Female,3,59,0,18,0,0,0,276,250.01,...,No,Up,No,No,No,No,No,Ch,Yes,>30
3,Male,2,44,1,16,0,0,0,8,250.43,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [26]:
df = df[df['gender'].isin(['Male', 'Female', 'male', 'female'])]
print(df['gender'].unique())

['Female' 'Male']


### 5.2 Create new Feature

In [27]:
df['total_hospital_procedures'] = df['num_lab_procedures'] + df['num_procedures']
df['med_to_procedure_ratio'] = df['num_medications'] / (df['total_hospital_procedures'] + 1)
df['total_historical_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df['severity_index_score'] = (df['number_outpatient'] * 1) + (df['number_emergency'] * 2) + (df['number_inpatient'] * 3)
df['has_history_of_emergency'] = (df['number_emergency'] > 0).astype(int)

In [28]:
def map_icd9(code):
    if code is None or str(code).strip() in ['?', 'None', 'nan', '']:
        return 'Unknown'
    code_str = str(code).split('.')[0].upper().strip()
    
    if code_str.startswith('V'):
        return 'Supplementary'
    if code_str.startswith('E'):
        return 'External_Injury'
    try:
        code_num = float(code_str)
        if 390 <= code_num <= 459 or code_num == 785:
            return 'Circulatory'
        elif 460 <= code_num <= 519 or code_num == 786:
            return 'Respiratory'
        elif 520 <= code_num <= 579 or code_num == 787:
            return 'Digestive'
        elif int(code_num) == 250:
            return 'Diabetes'
        elif 800 <= code_num <= 999:
            return 'Injury'
        elif 710 <= code_num <= 739:
            return 'Musculoskeletal'
        elif 580 <= code_num <= 629 or code_num == 788:
            return 'Genitourinary'
        elif 140 <= code_num <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

df['diag_1_group'] = df['diag_1'].apply(map_icd9)
df['diag_2_group'] = df['diag_2'].apply(map_icd9)
df['diag_3_group'] = df['diag_3'].apply(map_icd9)
df = df.drop(columns=['diag_1','diag_2','diag_3'],errors='ignore')

In [29]:
med_columns = [
    'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
    'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone',
    'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'insulin', 'glyburide-metformin', 'glipizide-metformin', 
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

df['num_diabetic_meds'] = df[med_columns].apply(lambda row: sum(row != 'no'), axis=1) # Hitung berapa banyak obat yang nilainya BUKAN 'no'
df['dosage_changes_count'] = df[med_columns].apply(lambda row: sum(row.isin(['up', 'down'])), axis=1) # Hitung berapa kali obat pasien mengalami status 'up' atau 'down'

In [30]:
def get_insulin_feature(row):  # Cek apakah mengonsumsi obat non-insulin
    other_meds = any(row[col] != 'no' for col in med_columns if col != 'insulin')
    insulin_status = row['insulin']
    
    if insulin_status != 'no' and not other_meds:
        return 'insulin_only'
    elif insulin_status != 'no' and other_meds:
        return 'combination_with_insulin'
    elif insulin_status == 'no' and other_meds:
        return 'other_meds_only'
    else:
        return 'no_meds'
df['insulin_treatment_type'] = df.apply(get_insulin_feature, axis=1)

In [31]:
med_map = {'No': 0, 'Down': 1, 'Steady': 2, 'Up': 3}
df_temp = df[med_columns].replace(med_map)
df['medication_intensity_score'] = df_temp.sum(axis=1) # Total skor intensitas obat untuk setiap pasien

C:\Users\HP\AppData\Local\Temp\ipykernel_17984\2835127022.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_temp = df[med_columns].replace(med_map)


In [32]:
df[y_target] = df[y_target].replace(['<30','>30'],'readmitted')
df[y_target].unique()

array(['readmitted', 'NO'], dtype=object)

In [33]:
df.head()

,gender,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,metformin,repaglinide,...,total_historical_visits,severity_index_score,has_history_of_emergency,diag_1_group,diag_2_group,diag_3_group,num_diabetic_meds,dosage_changes_count,insulin_treatment_type,medication_intensity_score
1,Female,3,59,0,18,0,0,0,No,No,...,0,0,0,Other,Diabetes,Other,22,0,combination_with_insulin,3
3,Male,2,44,1,16,0,0,0,No,No,...,0,0,0,Other,Diabetes,Circulatory,22,0,combination_with_insulin,3
4,Male,1,51,0,8,0,0,0,No,No,...,0,0,0,Neoplasms,Neoplasms,Diabetes,22,0,combination_with_insulin,4
6,Male,4,70,1,21,0,0,0,Steady,No,...,0,0,0,Circulatory,Circulatory,Supplementary,22,0,combination_with_insulin,4
7,Male,5,73,0,12,0,0,0,No,No,...,0,0,0,Circulatory,Respiratory,Diabetes,22,0,combination_with_insulin,2


## 6.Save Cleaned Dataset

In [34]:
file_path = 'dataset/diabetic_Binnary_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,gender,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,metformin,repaglinide,...,total_historical_visits,severity_index_score,has_history_of_emergency,diag_1_group,diag_2_group,diag_3_group,num_diabetic_meds,dosage_changes_count,insulin_treatment_type,medication_intensity_score
1,Female,3,59,0,18,0,0,0,No,No,...,0,0,0,Other,Diabetes,Other,22,0,combination_with_insulin,3
3,Male,2,44,1,16,0,0,0,No,No,...,0,0,0,Other,Diabetes,Circulatory,22,0,combination_with_insulin,3
4,Male,1,51,0,8,0,0,0,No,No,...,0,0,0,Neoplasms,Neoplasms,Diabetes,22,0,combination_with_insulin,4
6,Male,4,70,1,21,0,0,0,Steady,No,...,0,0,0,Circulatory,Circulatory,Supplementary,22,0,combination_with_insulin,4
7,Male,5,73,0,12,0,0,0,No,No,...,0,0,0,Circulatory,Respiratory,Diabetes,22,0,combination_with_insulin,2
